<a href="https://colab.research.google.com/github/salsabielmesl/flyrank-ml-assignments/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/salsabielmesl/flyrank-ml-assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import duckdb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

con = duckdb.connect()
dataset_path = '../../data/raw/content_refresh_anonymized.csv'

print("Setup complete. Data connection established.")

Setup complete. Data connection established.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis (Grain): One row represents one unique published content page (content_id) evaluated at a single decision point.
Time Window: Historical performance observed across a 90-day lookback window up to a mid-panel decision month (month = 2026-03).

In [6]:
!git clone https://github.com/salsabielmesl/flyrank-ml-assignments.git repo
%cd repo/work/notebooks

Cloning into 'repo'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 165 (delta 66), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 1.85 MiB | 2.85 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/repo/work/notebooks/repo/work/notebooks


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_base = con.execute(f"""
    SELECT *,
        CASE WHEN LOWER(trend_direction) = 'down' THEN 1 ELSE 0 END AS is_declining_label
    FROM '{dataset_path}'
""").df()

print("Target Distribution (1 = Declining, 0 = Stable/Growing):")
print(df_base['is_declining_label'].value_counts(normalize=True))

Target Distribution (1 = Declining, 0 = Stable/Growing):
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (Knowable at decision moment): days_since_last_update, impressions_90d, ctr, avg_position, word_count.
Label (Target to predict): is_declining_label (1 if trend_direction == 'down', 0 otherwise).
Context (Metadata): content_id, client_id, content_type, main_intent.
Excluded Variable: trend_pct is deliberately excluded because it encodes the percentage magnitude of traffic changes, causing catastrophic target leakage during training.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
null_target_count = df_base['is_declining_label'].isnull().sum()
print(f"Total Records: {len(df_base)}")
print(f"Missing Values in Target: {null_target_count}")

Total Records: 30000
Missing Values in Target: 0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

feat_days_since_update: Knowable at decision moment because it is calculated from the last CMS publication timestamp prior to evaluation.
feat_impressions_90d: Knowable at decision moment because it summarizes Search Console impression logs from the preceding 90 days.
feat_ctr: Knowable at decision moment because it uses historical click and impression aggregates recorded prior to evaluation.
feat_avg_position: Knowable at decision moment because search rankings are logged daily up to the evaluation date.
feat_word_count_k: Knowable at decision moment because current published article length is fetched from the database at evaluation time.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1_grain = con.execute(f"""
    SELECT content_id, COUNT(*) as cnt
    FROM '{dataset_path}'
    GROUP BY content_id HAVING COUNT(*) > 1
""").df()
print(f"1. Duplicate content_id count: {len(q1_grain)} (Must be 0)")

q2_span = con.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        MIN(content_age_days) as min_age_days,
        MAX(content_age_days) as max_age_days
    FROM '{dataset_path}'
""").df()
print("\n2. Row Count & Content Age Span:")
print(q2_span)

q3_avail = con.execute(f"""
    SELECT
        COUNT(*) as surviving_rows,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM '{dataset_path}')) as survival_pct
    FROM '{dataset_path}'
    WHERE (impressions_90d > 0) IS TRUE
      AND (avg_position > 0) IS TRUE
""").df()
print("\n3. Availability Check (IS TRUE):")
print(q3_avail)

df_features = con.execute(f"""
    SELECT
        days_since_last_update AS feat_days_since_update,
        impressions_90d AS feat_impressions_90d,
        ctr AS feat_ctr,
        avg_position AS feat_avg_position,
        (word_count / 1000.0) AS feat_word_count_k,
        trend_pct AS LEAKED_FEATURE,
        CASE WHEN LOWER(trend_direction) = 'down' THEN 1 ELSE 0 END AS is_declining_label
    FROM '{dataset_path}'
""").df()

y = df_features['is_declining_label']

X_leaked = df_features.drop(columns=['is_declining_label'])
clf_leak = RandomForestClassifier(random_state=42).fit(X_leaked, y)
score_leak = roc_auc_score(y, clf_leak.predict_proba(X_leaked)[:, 1])
print(f"\n🚨 LEAKED Model ROC-AUC Score: {score_leak:.4f}")

X_honest = df_features.drop(columns=['is_declining_label', 'LEAKED_FEATURE'])
clf_honest = RandomForestClassifier(random_state=42).fit(X_honest, y)
score_honest = roc_auc_score(y, clf_honest.predict_proba(X_honest)[:, 1])
print(f"✅ HONEST Model ROC-AUC Score: {score_honest:.4f}")

1. Duplicate content_id count: 0 (Must be 0)

2. Row Count & Content Age Span:
   total_rows  min_age_days  max_age_days
0       30000            90           564

3. Availability Check (IS TRUE):
   surviving_rows  survival_pct
0           28795     95.983333

🚨 LEAKED Model ROC-AUC Score: 1.0000
✅ HONEST Model ROC-AUC Score: 1.0000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named Data Limitation:
Seasonal Keyword Volatility: Search Console data reflects external search volume fluctuations. An observed downward trend during an off-peak month (e.g., March) may represent seasonal search volume drops rather than outdated content quality.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirmation output
print("Data contract verification complete.")

Data contract verification complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.